# 诗歌生成

# 数据处理

In [ ]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [ ]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        inp_emb = self.embed_layer(inp_ids)
        rnn_out = self.rnn_layer(inp_emb)
        logits = self.dense(rnn_out)
        return logits
    
    @tf.function
    def get_next_token(self, x, state, temperature=1.0):
        '''
        shape(x) = [b_sz,] 
        '''
    
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        
        # 引入温度采样，替代原来的 out = tf.argmax(logits, axis=-1)
        logits = logits / temperature
        out = tf.random.categorical(logits, num_samples=1)
        out = tf.squeeze(out, axis=-1)
        out = tf.cast(out, tf.int32)
        
        return out, state


## 一个计算sequence loss的辅助函数，只需了解用途。

In [ ]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [ ]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    losses = []
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)
        losses.append(loss.numpy())

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return losses


# 训练优化过程

In [ ]:
import matplotlib.pyplot as plt

optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
# Keep runtime reasonable for automated execution (nbconvert).
train_ds = train_ds.take(500)
model = myRNNModel(word2id)

all_losses = []
for epoch in range(1):
    losses = train(epoch, model, optimizer, train_ds)
    all_losses.extend(losses)
    
plt.figure(figsize=(10, 5))
plt.plot(all_losses, label='Training Loss')
plt.title('Training Loss over Steps')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


# 生成过程

In [ ]:
def gen_sentence(start_word=None, temperature=1.0):
    state = [tf.zeros((1, 128), dtype=tf.float32)]
    if start_word is None:
        cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    else:
        cur_token = tf.constant([word2id.get(start_word, word2id['UNK'])], dtype=tf.int32)
    
    collect = []
    punctuations = set(['，', '。', '？', '！', ',', '.'])
    
    # 加上给定的 start_word，我们只需再连续生成6个非标点汉字，凑齐七言
    for _ in range(6):
        for _ in range(50):  # 失败重试机制，避免死循环
            next_token, next_state = model.get_next_token(cur_token, state, temperature=tf.constant(temperature, dtype=tf.float32))
            t = int(next_token.numpy()[0])
            w = id2word.get(t, 'UNK')
            
            # 如果预测出的是标点或特殊字符则拒绝，并直接进行下一次采样尝试
            # 只有预测出汉字才会更新 cur_token 和状态 state
            if w not in punctuations and w not in ('PAD', 'UNK', start_token, end_token):
                cur_token = next_token
                state = next_state
                collect.append(w)
                break
                
    return collect

def gen_acrostic(head_chars, temperature=1.0):
    lines = []
    for i, ch in enumerate(head_chars):
        # 每次固定生成一句包含7个中文字符的句子
        words = gen_sentence(start_word=ch, temperature=temperature)
        
        # 在句子末尾加上规范的标点符号，奇数行逗号，偶数行句号
        punc = '，' if i % 2 == 0 else '。'
        lines.append(ch + ''.join(words) + punc)
    return '\n'.join(lines)

# 测试生成一首严格限制格式的七言藏头诗
print(gen_acrostic('日红山夜湖海月', temperature=1.0))

In [ ]:
def gen_sentence():
    state = [tf.random.normal(shape=(1, 128), stddev=0.5), tf.random.normal(shape=(1, 128), stddev=0.5)]
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        collect.append(cur_token.numpy()[0])
    return [id2word[t] for t in collect]
print(''.join(gen_sentence()))